# YOLO Head Detector Fine-Tuning on Colab A100

This notebook trains a YOLO head detector on the real labeled **RPEE-Heads** dataset and downloads the trained model at the end.

It is designed to work even if your uploaded project zip does not contain the local helper scripts. Dataset preparation, validation, training, evaluation, and packaging are done inside the notebook.

Use `Runtime > Change runtime type > GPU`, ideally **A100**.

In [ ]:
# 1. Confirm CUDA GPU
import torch

print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU found. In Colab, change runtime type to GPU before training.')
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

## 2. Load Your Project Zip

Upload `Crowd-analysis-main.zip` or any zip containing the project files. This check only requires the base repo files, not the new helper scripts.

In [ ]:
# 2. Load project from GitHub URL or uploaded zip
from pathlib import Path
import os
import shutil
import subprocess

PROJECT_DIR = Path('/content/crowd-analysis')
GITHUB_REPO_URL = ''  # Optional: set this if you prefer git clone over zip upload

def has_base_project_files(path: Path) -> bool:
    return (
        (path / 'AGENTS.md').exists()
        and (path / 'requirements.txt').exists()
        and (path / 'src').is_dir()
        and (path / 'configs').is_dir()
    )

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if GITHUB_REPO_URL.strip():
    subprocess.run(['git', 'clone', GITHUB_REPO_URL.strip(), str(PROJECT_DIR)], check=True)
else:
    from google.colab import files
    print('Upload a zip of the crowd-analysis project directory now.')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise RuntimeError('No .zip file uploaded.')

    upload_zip = Path('/content') / zip_names[0]
    extract_dir = Path('/content/project_upload')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(upload_zip), str(extract_dir))

    candidates = []
    if has_base_project_files(extract_dir):
        candidates.append(extract_dir)
    candidates.extend([p for p in extract_dir.rglob('*') if p.is_dir() and has_base_project_files(p)])

    print('Top-level extracted entries:', [p.name for p in extract_dir.iterdir()][:20])
    if not candidates:
        raise RuntimeError('Could not find base project files in uploaded zip. It must include AGENTS.md, requirements.txt, src/, and configs/.')
    shutil.copytree(candidates[0], PROJECT_DIR)

if not has_base_project_files(PROJECT_DIR):
    raise RuntimeError(f'Project setup failed: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
print('Project root:', Path.cwd())
!find . -maxdepth 2 -type f | sort | sed -n '1,80p'

In [ ]:
# 3. Install dependencies
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt

import ultralytics
print('Ultralytics:', ultralytics.__version__)

## 4. Download and Prepare RPEE-Heads

This downloads the real RPEE-Heads archive, keeps the official train/val split, holds out test, and writes `configs/head_dataset.yaml` for YOLO.

In [ ]:
# 4. Download real RPEE-Heads data
from pathlib import Path
import subprocess

RAW_DIR = Path('data/head_datasets/raw/rpee_heads')
ZIP_PATH = RAW_DIR / 'rpee_heads_dataset.zip'
RPEE_URL = 'https://ped.fz-juelich.de/data/machine_learning/2024_11_Recognition_In_Field_Studies/data/2024rpee_heads_dataset.zip'

RAW_DIR.mkdir(parents=True, exist_ok=True)
has_raw_images = any(RAW_DIR.rglob('*.jpg')) or any(RAW_DIR.rglob('*.jpeg')) or any(RAW_DIR.rglob('*.png'))
if not has_raw_images:
    print('Downloading RPEE-Heads (~1.1 GB)...')
    subprocess.run(['curl', '-L', '--fail', '--retry', '3', '-o', str(ZIP_PATH), RPEE_URL], check=True)
    print('Extracting RPEE-Heads...')
    subprocess.run(['python', '-m', 'zipfile', '-e', str(ZIP_PATH), str(RAW_DIR)], check=True)
else:
    print('Raw RPEE-Heads images already found; skipping download.')

print('Raw image count:', sum(1 for _ in RAW_DIR.rglob('*.jpg')) + sum(1 for _ in RAW_DIR.rglob('*.jpeg')) + sum(1 for _ in RAW_DIR.rglob('*.png')))

In [ ]:
# 5. Prepare YOLO train/val layout inline, with strict label checks
from pathlib import Path
import os
import shutil
import yaml

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
OUT_DIR = Path('data/head_datasets/yolo/rpee_heads')

def classify_split(path: Path):
    parts = [p.lower() for p in path.parts]
    joined = '/'.join(parts)
    if 'test' in parts or 'testing' in parts or 'test' in joined:
        return 'test'
    if 'val' in parts or 'validation' in parts or 'valid' in parts or 'val' in joined:
        return 'val'
    if 'train' in parts or 'training' in parts or 'train' in joined:
        return 'train'
    return None

def find_label(image: Path):
    direct = image.with_suffix('.txt')
    if direct.exists():
        return direct
    parts = list(image.parts)
    for i in range(len(parts) - 1, -1, -1):
        if parts[i].lower() in {'images', 'image', 'imgs', 'jpegimages'}:
            mirrored = list(parts)
            mirrored[i] = 'labels'
            candidate = Path(*mirrored).with_suffix('.txt')
            if candidate.exists():
                return candidate
    matches = list(RAW_DIR.rglob(image.with_suffix('.txt').name))
    return matches[0] if len(matches) == 1 else None

def clean_label(label_path: Path):
    kept = []
    bad = []
    for line_no, raw in enumerate(label_path.read_text(errors='replace').splitlines(), 1):
        line = raw.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            bad.append((label_path, line_no, line, 'field_count'))
            continue
        try:
            _cls = int(float(parts[0]))
            x, y, w, h = [float(v) for v in parts[1:]]
        except ValueError:
            bad.append((label_path, line_no, line, 'numeric'))
            continue
        if not all(0 <= v <= 1 for v in (x, y, w, h)) or w <= 0 or h <= 0:
            bad.append((label_path, line_no, line, 'normalized_or_size'))
            continue
        kept.append(f'0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}')
    return kept, bad

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
for split in ['train', 'val']:
    (OUT_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUT_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)

stats = {'train': {'images': 0, 'labels': 0, 'boxes': 0}, 'val': {'images': 0, 'labels': 0, 'boxes': 0}}
missing_labels = []
invalid_rows = []

images = sorted([p for p in RAW_DIR.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS])
if not images:
    raise RuntimeError('No real RPEE-Heads images found after download/extract.')

for image in images:
    split = classify_split(image.relative_to(RAW_DIR))
    if split == 'test':
        continue
    if split not in {'train', 'val'}:
        continue
    label = find_label(image)
    if label is None:
        missing_labels.append(str(image))
        continue
    kept, bad = clean_label(label)
    invalid_rows.extend(bad)
    image_dest = OUT_DIR / 'images' / split / image.name
    rel = os.path.relpath(image.resolve(), image_dest.parent.resolve())
    os.symlink(rel, image_dest)
    (OUT_DIR / 'labels' / split / f'{image.stem}.txt').write_text('\n'.join(kept) + ('\n' if kept else ''))
    stats[split]['images'] += 1
    stats[split]['labels'] += 1
    stats[split]['boxes'] += len(kept)

if missing_labels:
    raise RuntimeError(f'Missing labels for {len(missing_labels)} images. Example: {missing_labels[:3]}')
if stats['train']['boxes'] == 0 or stats['val']['boxes'] == 0:
    raise RuntimeError(f'No labeled boxes found in train/val: {stats}')

Path('configs').mkdir(exist_ok=True)
dataset_yaml = {
    'path': str(OUT_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'head'},
}
Path('configs/head_dataset.yaml').write_text(yaml.safe_dump(dataset_yaml, sort_keys=False))

print('Prepared stats:', stats)
print('Invalid rows skipped:', len(invalid_rows))
print(Path('configs/head_dataset.yaml').read_text())
assert stats['train']['images'] > 0 and stats['val']['images'] > 0
assert stats['train']['boxes'] > 0 and stats['val']['boxes'] > 0

In [ ]:
# 6. Train with Ultralytics YOLO
from pathlib import Path
import shutil
from ultralytics import YOLO

BASE_MODEL = 'yolo11n.pt'   # For better accuracy on A100, try 'yolo11s.pt'
EPOCHS = 25
IMAGE_SIZE = 640
BATCH_SIZE = 32             # Lower to 16 or 8 if CUDA OOM occurs
PROJECT_OUT = (Path.cwd() / 'models/fine_tuned').resolve()
RUN_NAME = 'head_detector'

model = YOLO(BASE_MODEL)
results = model.train(
    data=str((Path.cwd() / 'configs/head_dataset.yaml').resolve()),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    project=str(PROJECT_OUT),
    name=RUN_NAME,
    device=0,
    workers=8,
    exist_ok=True,
)

expected_best = PROJECT_OUT / RUN_NAME / 'weights/best.pt'
fallback_candidates = [
    expected_best,
    Path('models/fine_tuned/head_detector/weights/best.pt'),
    Path('runs/detect/models/fine_tuned/head_detector/weights/best.pt'),
    Path('/content/crowd-analysis/runs/detect/models/fine_tuned/head_detector/weights/best.pt'),
]
fallback_candidates.extend(sorted(Path.cwd().glob('**/head_detector/weights/best.pt')))

source_best = next((path for path in fallback_candidates if path.exists()), None)
if source_best is None:
    searched = '
'.join(str(path) for path in fallback_candidates[:20])
    raise RuntimeError(f'Training finished, but best.pt was not found. Checked:
{searched}')

# Normalize the artifact to the project path expected by later cells.
expected_best.parent.mkdir(parents=True, exist_ok=True)
if source_best.resolve() != expected_best.resolve():
    shutil.copy2(source_best, expected_best)

BEST_MODEL = expected_best
print('Best model source:', source_best.resolve())
print('Best model normalized path:', BEST_MODEL.resolve())


In [ ]:
# 7. Show metrics and output files
from pathlib import Path
import shutil
import pandas as pd

normalized_run = Path('models/fine_tuned/head_detector')
original_run = Path('runs/detect/models/fine_tuned/head_detector')

# If Ultralytics saved under runs/detect, copy the useful run artifacts into the expected folder.
if original_run.exists():
    normalized_run.mkdir(parents=True, exist_ok=True)
    for item in original_run.iterdir():
        target = normalized_run / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        else:
            shutil.copy2(item, target)

results_candidates = [
    normalized_run / 'results.csv',
    original_run / 'results.csv',
]
results_csv = next((path for path in results_candidates if path.exists()), None)

if results_csv is not None:
    df = pd.read_csv(results_csv)
    display(df.tail())
    print('Metrics CSV:', results_csv)
else:
    print('No results.csv found. Checked:', [str(p) for p in results_candidates])

!find models/fine_tuned/head_detector -maxdepth 2 -type f | sort | sed -n '1,120p'


In [ ]:
# 8. Evaluate on sample.mp4 if available; otherwise create a small validation slideshow video
from pathlib import Path
import cv2
import yaml
from ultralytics import YOLO

SOURCE_VIDEO = Path('data/input_videos/sample.mp4')
OUTPUT_VIDEO = Path('data/outputs/head_demo.mp4')

if not SOURCE_VIDEO.exists():
    print('sample.mp4 not found; creating a validation-image slideshow for inference demo.')
    cfg = yaml.safe_load(Path('configs/head_dataset.yaml').read_text())
    val_dir = Path(cfg['path']) / cfg['val']
    image_paths = sorted([p for p in val_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])[:120]
    if not image_paths:
        raise RuntimeError('No validation images found for fallback demo video.')
    SOURCE_VIDEO.parent.mkdir(parents=True, exist_ok=True)
    width, height = 1280, 720
    writer = cv2.VideoWriter(str(SOURCE_VIDEO), cv2.VideoWriter_fourcc(*'mp4v'), 12.0, (width, height))
    for image_path in image_paths:
        frame = cv2.imread(str(image_path))
        if frame is None:
            continue
        writer.write(cv2.resize(frame, (width, height)))
    writer.release()

detector = YOLO(str(BEST_MODEL))
cap = cv2.VideoCapture(str(SOURCE_VIDEO))
if not cap.isOpened():
    raise RuntimeError(f'Unable to open source video: {SOURCE_VIDEO}')

fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
OUTPUT_VIDEO.parent.mkdir(parents=True, exist_ok=True)
writer = cv2.VideoWriter(str(OUTPUT_VIDEO), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

frames = 0
total_heads = 0
max_heads = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    pred = detector.predict(frame, conf=0.25, iou=0.5, imgsz=640, device=0, verbose=False)[0]
    count = 0 if pred.boxes is None else len(pred.boxes)
    total_heads += count
    max_heads = max(max_heads, count)
    writer.write(pred.plot())
    frames += 1

cap.release()
writer.release()
assert OUTPUT_VIDEO.exists(), f'Missing output video: {OUTPUT_VIDEO}'
print('Evaluation source:', SOURCE_VIDEO)
print('Frames:', frames)
print('Average heads/frame:', total_heads / max(frames, 1))
print('Max heads/frame:', max_heads)
print('Output video:', OUTPUT_VIDEO)
!ls -lh data/outputs/head_demo.mp4

In [ ]:
# 9. Write training report and package downloadable artifacts
from pathlib import Path
import zipfile
import pandas as pd
import torch

REPORT = Path('docs/HEAD_MODEL_TRAINING_REPORT.md')
REPORT.parent.mkdir(parents=True, exist_ok=True)
results_csv = Path('models/fine_tuned/head_detector/results.csv')
metrics_text = 'No metrics CSV found.'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    if len(df):
        last = df.tail(1).to_dict(orient='records')[0]
        metrics_text = '\n'.join(f'- {k}: {v}' for k, v in last.items())

gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
REPORT.write_text(f'''# Head Model Training Report

## Dataset Used
RPEE-Heads was used. No backup dataset and no synthetic data were used.

## Training Configuration
- base model: {BASE_MODEL}
- epochs: {EPOCHS}
- image size: {IMAGE_SIZE}
- batch size: {BATCH_SIZE}
- device used: CUDA GPU 0 ({gpu_name})
- dataset YAML: `configs/head_dataset.yaml`
- output path: `models/fine_tuned/head_detector/`

## Training Result
- training completed: {'YES' if BEST_MODEL.exists() else 'NO'}
- best model path: `{BEST_MODEL}`
- metrics available: {'YES' if results_csv.exists() else 'NO'}

{metrics_text}

## Evaluation on Sample Video
- head_demo.mp4 created: {'YES' if OUTPUT_VIDEO.exists() else 'NO'}
- source video used: `{SOURCE_VIDEO}`
- output video path: `{OUTPUT_VIDEO}`
- observations: automated inference completed; visual review is still required.
- limitations: RPEE-Heads is not Indian Railway CCTV. Domain validation on approved Indian railway footage is still required.

## Next Steps
- validate on Indian Railway CCTV
- optionally train longer
- optionally use yolo11s
- tune thresholds
- compare with body detector
''', encoding='utf-8')
print(REPORT.read_text())

artifact_zip = Path('/content/head_detector_colab_outputs.zip')
artifact_paths = [
    BEST_MODEL,
    Path('models/fine_tuned/head_detector/weights/last.pt'),
    results_csv,
    Path('models/fine_tuned/head_detector/args.yaml'),
    REPORT,
    OUTPUT_VIDEO,
]
with zipfile.ZipFile(artifact_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in artifact_paths:
        if path.exists():
            zf.write(path, arcname=str(path))
print('Artifact zip:', artifact_zip)

In [ ]:
# 10. Download best.pt and the artifact bundle
from google.colab import files

files.download(str(BEST_MODEL))
files.download('/content/head_detector_colab_outputs.zip')
print('After the run finishes, also use Colab: File > Download > Download .ipynb, then paste/share that executed notebook back here.')